# Stage 3 - DPO Preference Alignment

**Goal:** push the SFT model to prefer *correct, specific* answers over *wrong / generic* ones
(hallucinated tables, missing DISTINCT, lazy SELECT *, wrong joins).

Data: `ecomm-db-preference` on the Hugging Face Hub (63 `{prompt, chosen, rejected}` examples).

In [ ]:
# Install Unsloth (needs a CUDA GPU). If Colab prompts to restart after install, do it.
%%capture
!pip install --upgrade unsloth unsloth_zoo
# Ensure a MODERN TRL (provides SFTConfig/DPOConfig + processing_class) that matches new transformers.
!pip install --upgrade "trl>=0.13.1"

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > Change runtime type > T4 GPU")

## 1. Config

In [ ]:
# --- Datasets live on the Hugging Face Hub (pushed via scripts/push_to_hf.py) ---
HF_USER = "Rajesh507"   # <-- your Hugging Face username
DS_NONINSTRUCT = f"{HF_USER}/ecomm-db-noninstruct"
DS_INSTRUCTION = f"{HF_USER}/ecomm-db-instruction"
DS_PREFERENCE  = f"{HF_USER}/ecomm-db-preference"

# If the datasets are PRIVATE, log in first (needs a read token):
# from huggingface_hub import login; login()

In [ ]:
# Persist trained adapters to the Hugging Face Hub (no Google Drive needed).
from huggingface_hub import login
login()   # paste a WRITE token: https://huggingface.co/settings/tokens

ADAPTER_STAGE1 = f"{HF_USER}/ecomm-db-stage1-noninstruct"
ADAPTER_STAGE2 = f"{HF_USER}/ecomm-db-stage2-sft"
ADAPTER_STAGE3 = f"{HF_USER}/ecomm-db-stage3-dpo"

# Merged 16-bit models that chain the stages: Stage N merges its LoRA into the
# base, and Stage N+1 loads that merged model and adds a FRESH, fully-trainable
# LoRA on top (bootcamp Class-22's "merge then add new adapter" approach).
MERGED_STAGE1 = f"{HF_USER}/ecomm-db-stage1-merged"
MERGED_STAGE2 = f"{HF_USER}/ecomm-db-stage2-merged"
print("Adapters will be pushed under:", HF_USER)

## 2. Load the merged SFT model (Stage 2) from the Hugging Face Hub

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-Coder-1.5B"   # Coder variant is stronger at SQL. Alt: "unsloth/Qwen2.5-1.5B"
MAX_SEQ_LEN = 2048

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MERGED_STAGE2,   # merged SFT model from Stage 2 (fresh LoRA added below)
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)

In [ ]:
from unsloth import FastLanguageModel

# If the loaded model ALREADY has a LoRA adapter (i.e. we continued from a
# previous stage's adapter), keep training THAT adapter instead of adding a new,
# conflicting one. Otherwise (a fresh base model) attach a new LoRA adapter.
if getattr(model, "peft_config", None):
    print("Model already has a LoRA adapter - continuing to train it.")
    try:
        FastLanguageModel.for_training(model)
    except Exception:
        pass
    # A loaded adapter is often frozen (inference mode); re-enable grads so
    # training actually updates it.
    for _n, _p in model.named_parameters():
        if "lora_" in _n:
            _p.requires_grad_(True)
else:
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,                       # LoRA rank
        lora_alpha = 16,              # scaling
        lora_dropout = 0,             # 0 is optimized in Unsloth
        bias = "none",
        target_modules = ["q_proj","k_proj","v_proj","o_proj",
                          "gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

# Sanity: there MUST be trainable parameters, else trainer.train() does nothing
# and the model stays generic. This catches a frozen/misloaded adapter early.
model.print_trainable_parameters()
_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert _trainable > 0, "No trainable parameters - adapter is frozen; training would do nothing."
print("Trainable params:", _trainable)

## 3. Load and format the preference dataset

In [ ]:
# A single, consistent prompt template used across ALL three stages.
PROMPT = """Below is a question about the client e-commerce database schema. \
Write a response that correctly answers it, giving the exact table name(s) or a valid SQL query.

### Question:
{}

### Answer:
{}"""

In [ ]:
from datasets import load_dataset
EOS = tokenizer.eos_token

def format_pref(ex):
    return {
        "prompt": PROMPT.format(ex["prompt"], ""),
        "chosen": ex["chosen"] + EOS,
        "rejected": ex["rejected"] + EOS,
    }

ds = load_dataset(DS_PREFERENCE, split="train").map(format_pref)
print(ds); print(ds[0])

## 4. Configure and run DPO

In [ ]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,          # LoRA -> no separate reference model needed (saves memory)
    processing_class = tokenizer,   # new TRL API (was 'tokenizer=')
    train_dataset = ds,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 5e-5,
        beta = 0.1,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 3407,
        max_length = MAX_SEQ_LEN,
        max_prompt_length = 512,
        output_dir = "outputs_stage3",
        report_to = "none",
    ),
)
dpo_trainer.train()

## 5. Test after DPO on the reloaded MERGED model
Save the trained model merged, reload it, and evaluate the reloaded model so generation always reflects the training (HR-project pattern).

In [ ]:
# Bake the trained LoRA into the weights, then RELOAD the merged model and
# evaluate THAT (not the in-memory adapter). See generator notes: this is the
# HR-project pattern that reliably reflects the training at generation time.
model.save_pretrained_merged("stage3_merged_local", tokenizer, save_method="merged_16bit")

eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "stage3_merged_local",
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = False,   # full 16-bit reload = exact recall of memorized answers
)
FastLanguageModel.for_inference(eval_model)

def ask(question, max_new_tokens=128):
    text = PROMPT.format(question, "")
    inputs = eval_tokenizer(text, return_tensors="pt").to("cuda")
    out = eval_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return eval_tokenizer.decode(out[0], skip_special_tokens=True).split("### Answer:")[-1].strip()

for q in [
    'Find the number of unique orders placed by customer 1001.',
    'Which table stores product images?',
    'Find the top 5 customers by total spend.',
]:
    print("Q:", q); print("A:", ask(q)); print("-"*60)

## 6. Push the DPO-aligned adapter to the Hugging Face Hub

In [ ]:
model.push_to_hub(ADAPTER_STAGE3, token=True)
tokenizer.push_to_hub(ADAPTER_STAGE3, token=True)
print("Pushed DPO adapter to:", ADAPTER_STAGE3)

# Optional: merge to a standalone 16-bit model and push it (no adapter needed to load)
# model.push_to_hub_merged(f"{HF_USER}/ecomm-db-final-merged", tokenizer, save_method='merged_16bit', token=True)

### Done
Three adapters are now on the Hugging Face Hub (Stage 1 / 2 / 3). Use `src/inference.py` (point `--adapter` at the DPO repo) and fill in the `reports/` comparison tables.